# Build and Train an Exact 10M JAX Addition Transformer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/jax-addition-transformer/blob/main/notebooks/01_build_train_exact_10m.ipynb)

A self-contained decoder-only transformer written with JAX, Flax NNX, and Optax. It learns addition from `000 + 000` through `999 + 999`.

The notebook uses one fixed architecture: **5 layers, width 320, 5 attention heads, GELU, Pre-LayerNorm, learned positions, AdamW, exactly 10,000,000 trainable parameters.**


## Setup — pinned Colab T4 runtime

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Run the environment installation cell below. It uses `uv` to create one
   isolated, pinned CUDA 12 environment ahead of Colab's preinstalled packages.
3. Colab restarts the Python process exactly once. After it reconnects, choose
   **Runtime → Run all** (or rerun from this setup cell).
4. Continue only when the validation cell prints `Backend: gpu`.

JAX, JAXlib, the CUDA plugin, and Flax are pinned together. An unbounded
`jax[cuda12]` install can combine JAX 0.11.0 with Flax 0.12.2 and fail at
`from flax import nnx` because JAX 0.11.0 removed `jax.core.Effect`.


In [ ]:
# Generated by scripts/sync_colab_runtime.py; do not hand-edit.
import importlib.metadata as _metadata
import importlib.util as _importlib_util
import json as _json
import os as _os
import platform as _platform
import shutil as _shutil
import signal as _signal
import site as _site
import subprocess as _subprocess
import sys as _sys
from pathlib import Path as _Path

_COLAB_FINGERPRINT = 'jax-0.8.1-flax-0.12.2-cuda12-v1'
_EXPECTED_PYTHON = '3.12'
_UV_BOOTSTRAP_VERSION = '0.11.28'
_REQUIREMENTS = [
    "jax[cuda12]==0.8.1",
    "flax==0.12.2",
    "optax==0.2.6",
    "numpy==2.3.3",
    "matplotlib==3.10.7"
]
_EXPECTED_VERSIONS = {
    "flax": "0.12.2",
    "jax": "0.8.1",
    "jax-cuda12-pjrt": "0.8.1",
    "jax-cuda12-plugin": "0.8.1",
    "jaxlib": "0.8.1",
    "matplotlib": "3.10.7",
    "numpy": "2.3.3",
    "optax": "0.2.6"
}
_IN_COLAB = _importlib_util.find_spec("google.colab") is not None
_ENV_ROOT = _Path("/content/.jax-addition-transformer-colab")
_ENV_PYTHON = _ENV_ROOT / "bin" / "python"
_ENV_SITE_PACKAGES = (
    _ENV_ROOT
    / "lib"
    / f"python{_sys.version_info.major}.{_sys.version_info.minor}"
    / "site-packages"
)
_RESTART_MARKER = _ENV_ROOT / f".{_COLAB_FINGERPRINT}.json"
_PTH_PATH = _Path(_site.getsitepackages()[0]) / "00-jax-addition-colab.pth"
_PTH_CONTENT = (
    "import sys; "
    f"sys.path.insert(0, {str(_ENV_SITE_PACKAGES)!r})\n"
)

if _ENV_SITE_PACKAGES.is_dir():
    _site.addsitedir(str(_ENV_SITE_PACKAGES))
    if str(_ENV_SITE_PACKAGES) in _sys.path:
        _sys.path.remove(str(_ENV_SITE_PACKAGES))
    _sys.path.insert(0, str(_ENV_SITE_PACKAGES))

print("Python:", _platform.python_version())
print("Operating system:", _platform.platform())
print("Google Colab:", _IN_COLAB)
_smi = _subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
    check=False,
    capture_output=True,
    text=True,
)
print("Accelerator:", _smi.stdout.strip() or "not detected")

if ".".join(map(str, _sys.version_info[:2])) != _EXPECTED_PYTHON:
    raise RuntimeError(
        f"This notebook requires Python {_EXPECTED_PYTHON}; "
        f"the active runtime is {_platform.python_version()}. "
        "Select Colab's latest runtime and reconnect."
    )
if _IN_COLAB and _smi.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU is attached. Select Runtime → Change runtime type → T4 GPU."
    )

def _installed_version(distribution):
    try:
        return _metadata.version(distribution)
    except _metadata.PackageNotFoundError:
        return None

_mismatches = {
    name: (_installed_version(name), expected)
    for name, expected in _EXPECTED_VERSIONS.items()
    if _installed_version(name) != expected
}

if _mismatches:
    if not _IN_COLAB:
        raise RuntimeError(
            "Pinned Colab packages are not active: "
            + _json.dumps(_mismatches, sort_keys=True)
            + "\nCreate an isolated environment from configs/colab-runtime.json."
        )
    if _RESTART_MARKER.exists():
        raise RuntimeError(
            "The pinned installation is still inconsistent after the controlled "
            "restart; refusing to restart again. Details: "
            + _json.dumps(_mismatches, sort_keys=True)
        )

    _uv = _shutil.which("uv")
    if _uv is None:
        print(f"Bootstrapping uv=={_UV_BOOTSTRAP_VERSION}...")
        _subprocess.run(
            [
                _sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                f"uv=={_UV_BOOTSTRAP_VERSION}",
            ],
            check=True,
        )
        _uv = _shutil.which("uv")
    if _uv is None:
        raise RuntimeError("uv was installed but its executable was not found.")

    print("Creating an isolated pinned JAX/Flax environment:", *_REQUIREMENTS)
    _subprocess.run(
        [
            _uv,
            "venv",
            "--clear",
            "--python",
            _sys.executable,
            str(_ENV_ROOT),
        ],
        check=True,
    )
    _subprocess.run(
        [
            _uv,
            "pip",
            "install",
            "--python",
            str(_ENV_PYTHON),
            "--upgrade",
            *_REQUIREMENTS,
        ],
        check=True,
    )
    _subprocess.run(
        [_uv, "pip", "check", "--python", str(_ENV_PYTHON)],
        check=True,
    )

    _metadata_check = (
        "import importlib.metadata as m, json; "
        f"expected={_EXPECTED_VERSIONS!r}; "
        "resolved={name: m.version(name) for name in expected}; "
        "assert resolved == expected, (resolved, expected); "
        "print(json.dumps(resolved, sort_keys=True))"
    )
    _subprocess.run(
        [_ENV_PYTHON, "-c", _metadata_check],
        check=True,
    )
    _PTH_PATH.write_text(_PTH_CONTENT)

    _RESTART_MARKER.write_text(
        _json.dumps(
            {"fingerprint": _COLAB_FINGERPRINT, "packages": _EXPECTED_VERSIONS},
            sort_keys=True,
        )
    )
    print(
        "Pinned packages installed and dependency checks passed. "
        "Colab will now restart once. After reconnecting, run all cells again."
    )
    _os.kill(_os.getpid(), _signal.SIGKILL)

if _IN_COLAB and _ENV_SITE_PACKAGES.is_dir():
    if not _PTH_PATH.is_file() or _PTH_PATH.read_text() != _PTH_CONTENT:
        _PTH_PATH.write_text(_PTH_CONTENT)

print("Pinned Colab environment is already installed; no restart is needed.")


In [ ]:
# COLAB_RUNTIME_VALIDATE_GPU_V1
_resolved_versions = {
    name: _metadata.version(name)
    for name in _EXPECTED_VERSIONS
}
assert _resolved_versions == _EXPECTED_VERSIONS, _resolved_versions

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx
import flax as _flax

assert jax.__version__ == _EXPECTED_VERSIONS["jax"]
assert _flax.__version__ == _EXPECTED_VERSIONS["flax"]
assert optax.__version__ == _EXPECTED_VERSIONS["optax"]
assert np.__version__ == _EXPECTED_VERSIONS["numpy"]

_backend = jax.default_backend()
_devices = jax.devices()
if _backend != "gpu" or not any(device.platform == "gpu" for device in _devices):
    raise RuntimeError(
        f"JAX backend is {_backend!r}, not 'gpu'. "
        "Select Runtime → Change runtime type → T4 GPU and run setup again."
    )
_matrix_product = jnp.ones((128, 128), dtype=jnp.float32) @ jnp.ones(
    (128, 128), dtype=jnp.float32
)
jax.block_until_ready(_matrix_product)
if _matrix_product.device.platform != "gpu":
    raise RuntimeError("The validation matrix multiplication did not execute on the GPU.")

_nnx_probe = nnx.Linear(4, 4, rngs=nnx.Rngs(0))
_nnx_graphdef, _nnx_state = nnx.split(_nnx_probe)
_nnx_restored = nnx.merge(_nnx_graphdef, _nnx_state)
assert _nnx_restored(jnp.ones((1, 4), dtype=jnp.float32)).shape == (1, 4)

print("JAX:", _resolved_versions["jax"])
print("JAXlib:", _resolved_versions["jaxlib"])
print("Flax:", _resolved_versions["flax"])
print("Optax:", _resolved_versions["optax"])
print("NumPy:", _resolved_versions["numpy"])
print("Backend:", _backend)
print("Device:", _devices[0].device_kind)
print("GPU matrix multiplication and Flax NNX split/merge: passed")


import csv
import json
import math
import pickle
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt


In [ ]:
SEED = 42

VOCAB_SIZE = 13
MAX_SEQUENCE_LENGTH = 16
MODEL_INPUT_LENGTH = 15
PROMPT_LENGTH = 12
ANSWER_DIGITS = 4

N_LAYERS = 5
D_MODEL = 320
N_HEADS = 5
HEAD_DIM = 64
D_FF = 2480

TRAIN_SIZE = 200_000
VALIDATION_SIZE = 20_000
TEST_SIZE = 780_000

BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 4000
MAX_STEPS = 5000
LOG_EVERY = 25
EVAL_EVERY = 250

PEAK_LR = 1e-3
FINAL_LR = 1e-4
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

PARAM_DTYPE = jnp.float32
COMPUTE_DTYPE = jnp.float16
RUN_DIR = Path("/content/jax_addition_run")
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RUN_DIR / "exact_10m_checkpoint.pkl"
HISTORY_PATH = RUN_DIR / "history.json"
FAILURES_PATH = RUN_DIR / "failures.csv"
RESULTS_PATH = RUN_DIR / "results.json"

assert D_MODEL == N_HEADS * HEAD_DIM
assert TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE == 1_000_000


## Data

Each example has one fixed 16-token layout:

```text
123 + 456 = 9750
```

`9750` is `0579` reversed. The model therefore generates the units digit first, then follows the carry direction. Loss is applied only to the four answer digits.


In [ ]:
TOKENS = "0123456789 +="
TOKEN_TO_ID = {token: index for index, token in enumerate(TOKENS)}
ID_TO_TOKEN = {index: token for token, index in TOKEN_TO_ID.items()}

def encode(text):
    assert all(character in TOKEN_TO_ID for character in text)
    return np.asarray([TOKEN_TO_ID[character] for character in text], dtype=np.int32)

def decode(token_ids):
    token_ids = np.asarray(token_ids)
    assert token_ids.ndim == 1
    assert np.all((0 <= token_ids) & (token_ids < VOCAB_SIZE))
    return "".join(ID_TO_TOKEN[int(token_id)] for token_id in token_ids)

def format_prompt(a, b):
    assert 0 <= a <= 999 and 0 <= b <= 999
    return f"{a:03d} + {b:03d} = "

def format_example(a, b):
    normal_answer = f"{a + b:04d}"
    return format_prompt(a, b) + normal_answer[::-1]

def decode_internal_answer(token_ids):
    text = decode(token_ids)
    assert len(text) == 4 and text.isdigit()
    return text[::-1].lstrip("0") or "0"

for a, b in [(0, 0), (7, 42), (99, 1), (123, 456), (999, 999)]:
    text = format_example(a, b)
    assert len(text) == MAX_SEQUENCE_LENGTH
    print(text)


In [ ]:
def pair_ids_to_operands(pair_ids):
    pair_ids = np.asarray(pair_ids, dtype=np.int32)
    return pair_ids // 1000, pair_ids % 1000

def operand_lengths(values):
    values = np.asarray(values)
    return np.where(values < 10, 1, np.where(values < 100, 2, 3)).astype(np.int8)

def carry_codes(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    carry_units = ((a % 10) + (b % 10) >= 10).astype(np.int8)
    carry_tens = (((a // 10) % 10) + ((b // 10) % 10) + carry_units >= 10).astype(np.int8)
    carry_hundreds = (
        ((a // 100) % 10) + ((b // 100) % 10) + carry_tens >= 10
    ).astype(np.int8)
    return (4 * carry_units + 2 * carry_tens + carry_hundreds).astype(np.int8)

def stratum_codes(a, b):
    return (
        ((operand_lengths(a) - 1) * 3 + (operand_lengths(b) - 1)) * 8
        + carry_codes(a, b)
    ).astype(np.int16)

def make_sequences(pair_ids):
    pair_ids = np.asarray(pair_ids, dtype=np.int32)
    a, b = pair_ids_to_operands(pair_ids)
    total = a + b

    sequences = np.empty((len(pair_ids), MAX_SEQUENCE_LENGTH), dtype=np.uint8)
    sequences[:, 0] = a // 100
    sequences[:, 1] = (a // 10) % 10
    sequences[:, 2] = a % 10
    sequences[:, 3:6] = (10, 11, 10)
    sequences[:, 6] = b // 100
    sequences[:, 7] = (b // 10) % 10
    sequences[:, 8] = b % 10
    sequences[:, 9:12] = (10, 12, 10)
    sequences[:, 12] = total % 10
    sequences[:, 13] = (total // 10) % 10
    sequences[:, 14] = (total // 100) % 10
    sequences[:, 15] = (total // 1000) % 10
    return sequences

def largest_remainder(counts, total, capacity):
    ideal = counts.astype(np.float64) * total / counts.sum()
    allocated = np.minimum(np.floor(ideal).astype(np.int64), capacity)
    order = np.lexsort((np.arange(len(counts)), -(ideal - allocated)))
    remaining = total - int(allocated.sum())
    while remaining:
        eligible = order[allocated[order] < capacity[order]]
        take = eligible[:remaining]
        allocated[take] += 1
        remaining -= len(take)
    return allocated

def build_split(seed=SEED):
    pair_ids = np.arange(1_000_000, dtype=np.int32)
    a, b = pair_ids_to_operands(pair_ids)
    codes = stratum_codes(a, b)
    unique_codes, counts = np.unique(codes, return_counts=True)

    train_counts = largest_remainder(counts, TRAIN_SIZE, counts)
    validation_counts = largest_remainder(
        counts, VALIDATION_SIZE, counts - train_counts
    )

    rng = np.random.default_rng(seed)
    train_parts = []
    validation_parts = []
    test_parts = []

    for code_value, train_count, validation_count in zip(
        unique_codes, train_counts, validation_counts, strict=True
    ):
        members = pair_ids[codes == code_value].copy()
        rng.shuffle(members)
        train_parts.append(members[:train_count])
        validation_parts.append(members[train_count : train_count + validation_count])
        test_parts.append(members[train_count + validation_count :])

    train_ids = np.concatenate(train_parts)
    validation_ids = np.concatenate(validation_parts)
    test_ids = np.concatenate(test_parts)
    rng.shuffle(train_ids)
    rng.shuffle(validation_ids)
    rng.shuffle(test_ids)

    assert len(train_ids) == TRAIN_SIZE
    assert len(validation_ids) == VALIDATION_SIZE
    assert len(test_ids) == TEST_SIZE
    assert len(np.unique(np.concatenate([train_ids, validation_ids, test_ids]))) == 1_000_000
    return train_ids, validation_ids, test_ids

train_ids, validation_ids, test_ids = build_split()
train_sequences = make_sequences(train_ids)
train_a, train_b = pair_ids_to_operands(train_ids)
train_strata = stratum_codes(train_a, train_b)

print(f"Train: {len(train_ids):,}")
print(f"Validation: {len(validation_ids):,}")
print(f"Test: {len(test_ids):,}")
print(f"Cached training data: {train_sequences.nbytes / 1e6:.1f} MB")


In [ ]:
class HybridBatcher:
    def __init__(self, sequences, strata, batch_size, seed):
        self.sequences = sequences
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.stratum_indices = [
            np.flatnonzero(strata == code)
            for code in np.unique(strata)
        ]

    def sample(self):
        natural_count = self.batch_size // 2
        balanced_count = self.batch_size - natural_count

        natural = self.rng.integers(0, len(self.sequences), size=natural_count)

        selected_strata = self.rng.integers(
            0, len(self.stratum_indices), size=balanced_count
        )
        balanced = np.empty(balanced_count, dtype=np.int64)

        for stratum in np.unique(selected_strata):
            locations = np.flatnonzero(selected_strata == stratum)
            balanced[locations] = self.rng.choice(
                self.stratum_indices[int(stratum)],
                size=len(locations),
                replace=True,
            )

        indices = np.concatenate([natural, balanced])
        self.rng.shuffle(indices)
        batch = self.sequences[indices].astype(np.int32)
        return batch[:, :-1], batch[:, 1:]

batcher = HybridBatcher(train_sequences, train_strata, BATCH_SIZE, SEED)

sample_inputs, sample_targets = batcher.sample()
assert sample_inputs.shape == (BATCH_SIZE, MODEL_INPUT_LENGTH)
assert sample_targets.shape == (BATCH_SIZE, MODEL_INPUT_LENGTH)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))

example = format_example(123, 456)
for position, token in enumerate(example):
    axes[0].add_patch(plt.Rectangle((position, 0), 0.9, 0.8, fill=False))
    axes[0].text(
        position + 0.45,
        0.4,
        "space" if token == " " else token,
        ha="center",
        va="center",
        fontsize=8,
    )
axes[0].set_xlim(0, 16)
axes[0].set_ylim(0, 1)
axes[0].axis("off")
axes[0].set_title("123 + 456 = 9750")

sample_pair_ids = train_ids[np.random.default_rng(SEED).integers(0, len(train_ids), 50_000)]
sample_a, sample_b = pair_ids_to_operands(sample_pair_ids)
natural_distribution = np.bincount(carry_codes(sample_a, sample_b), minlength=8) / 50_000

balanced_inputs, _ = batcher.sample()
balanced_a = balanced_inputs[:, 0] * 100 + balanced_inputs[:, 1] * 10 + balanced_inputs[:, 2]
balanced_b = balanced_inputs[:, 6] * 100 + balanced_inputs[:, 7] * 10 + balanced_inputs[:, 8]
balanced_distribution = np.bincount(
    carry_codes(balanced_a, balanced_b), minlength=8
) / BATCH_SIZE

x = np.arange(8)
axes[1].bar(x - 0.2, natural_distribution, 0.4, label="natural")
axes[1].bar(x + 0.2, balanced_distribution, 0.4, label="training batch")
axes[1].set_xticks(x, [f"{value:03b}" for value in x])
axes[1].set_xlabel("carry pattern: units, tens, hundreds")
axes[1].set_ylabel("fraction")
axes[1].set_title("Carry coverage")
axes[1].legend()

plt.tight_layout()
plt.show()


## Model


In [ ]:
def normal_parameter(rngs, shape, scale):
    values = jax.random.normal(rngs.params(), shape, dtype=jnp.float32) * scale
    return nnx.Param(values.astype(PARAM_DTYPE))

def matrix_multiply(x, weight):
    x16 = x.astype(COMPUTE_DTYPE)
    w16 = weight.astype(COMPUTE_DTYPE)
    return jax.lax.dot_general(
        x16,
        w16,
        (((x16.ndim - 1,), (0,)), ((), ())),
        preferred_element_type=jnp.float32,
    )

class Linear(nnx.Module):
    def __init__(self, input_width, output_width, rngs, scale=0.02):
        self.kernel = normal_parameter(rngs, (input_width, output_width), scale)

    def __call__(self, x):
        return matrix_multiply(x, self.kernel.get_value())

class LayerNorm(nnx.Module):
    def __init__(self, width):
        self.scale = nnx.Param(jnp.ones((width,), dtype=PARAM_DTYPE))
        self.bias = nnx.Param(jnp.zeros((width,), dtype=PARAM_DTYPE))

    def __call__(self, x):
        x = x.astype(jnp.float32)
        mean = jnp.mean(x, axis=-1, keepdims=True)
        variance = jnp.mean(jnp.square(x - mean), axis=-1, keepdims=True)
        normalized = (x - mean) * jax.lax.rsqrt(variance + 1e-5)
        return normalized * self.scale.get_value() + self.bias.get_value()

def gelu(x):
    return 0.5 * x * (1.0 + jax.lax.erf(x / math.sqrt(2.0)))

CAUSAL_MASK = jnp.tril(jnp.ones((MODEL_INPUT_LENGTH, MODEL_INPUT_LENGTH), dtype=bool))

class CausalMHA(nnx.Module):
    def __init__(self, rngs):
        residual_scale = 0.02 / math.sqrt(2 * N_LAYERS)
        self.q_proj = Linear(D_MODEL, D_MODEL, rngs)
        self.k_proj = Linear(D_MODEL, D_MODEL, rngs)
        self.v_proj = Linear(D_MODEL, D_MODEL, rngs)
        self.out_proj = Linear(D_MODEL, D_MODEL, rngs, scale=residual_scale)

    def __call__(self, x, return_attention=False):
        batch, length, _ = x.shape
        q = self.q_proj(x).reshape(batch, length, N_HEADS, HEAD_DIM)
        k = self.k_proj(x).reshape(batch, length, N_HEADS, HEAD_DIM)
        v = self.v_proj(x).reshape(batch, length, N_HEADS, HEAD_DIM)

        scores = jnp.einsum(
            "bthd,bshd->bhts",
            q.astype(jnp.float32),
            k.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        ) / math.sqrt(HEAD_DIM)

        scores = jnp.where(
            CAUSAL_MASK[None, None, :length, :length],
            scores,
            jnp.finfo(jnp.float32).min,
        )
        probabilities = jax.nn.softmax(scores, axis=-1)

        attended = jnp.einsum(
            "bhts,bshd->bthd",
            probabilities,
            v.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        )
        output = self.out_proj(attended.reshape(batch, length, D_MODEL))
        return (output, probabilities) if return_attention else output

class FeedForward(nnx.Module):
    def __init__(self, rngs):
        residual_scale = 0.02 / math.sqrt(2 * N_LAYERS)
        self.up = Linear(D_MODEL, D_FF, rngs)
        self.down = Linear(D_FF, D_MODEL, rngs, scale=residual_scale)

    def __call__(self, x):
        return self.down(gelu(self.up(x)))

class TransformerBlock(nnx.Module):
    def __init__(self, rngs):
        self.attention_norm = LayerNorm(D_MODEL)
        self.attention = CausalMHA(rngs)
        self.ffn_norm = LayerNorm(D_MODEL)
        self.ffn = FeedForward(rngs)

    def __call__(self, x, return_attention=False):
        attention_result = self.attention(
            self.attention_norm(x),
            return_attention=return_attention,
        )
        if return_attention:
            attended, probabilities = attention_result
        else:
            attended = attention_result

        x = x + attended
        x = x + self.ffn(self.ffn_norm(x))
        return (x, probabilities) if return_attention else x


In [ ]:
class AdditionTransformer(nnx.Module):
    def __init__(self, rngs):
        self.token_embedding = normal_parameter(rngs, (VOCAB_SIZE, D_MODEL), 0.02)
        self.position_embedding = normal_parameter(
            rngs, (MODEL_INPUT_LENGTH, D_MODEL), 0.02
        )
        self.blocks = nnx.List(
            [TransformerBlock(rngs) for _ in range(N_LAYERS)]
        )
        self.final_norm = LayerNorm(D_MODEL)

    def __call__(self, token_ids, return_attention=False):
        assert token_ids.ndim == 2
        assert token_ids.shape[1] == MODEL_INPUT_LENGTH

        x = self.token_embedding.get_value()[token_ids]
        x = x + self.position_embedding.get_value()[None, :, :]

        attention_maps = []
        for block in self.blocks:
            if return_attention:
                x, probabilities = block(x, return_attention=True)
                attention_maps.append(probabilities)
            else:
                x = block(x)

        x = self.final_norm(x)
        logits = jnp.einsum(
            "btd,vd->btv",
            x.astype(jnp.float32),
            self.token_embedding.get_value().astype(jnp.float32),
            preferred_element_type=jnp.float32,
        )
        return (logits, attention_maps) if return_attention else logits

model = AdditionTransformer(nnx.Rngs(params=SEED))
parameter_state = nnx.state(model, nnx.Param)
parameter_count = sum(int(leaf.size) for leaf in jax.tree.leaves(parameter_state))

parameter_rows = [
    ("attention projections", N_LAYERS * 4 * D_MODEL * D_MODEL),
    ("feed-forward networks", N_LAYERS * 2 * D_MODEL * D_FF),
    ("block LayerNorms", N_LAYERS * 4 * D_MODEL),
    ("token embedding / tied head", VOCAB_SIZE * D_MODEL),
    ("position embedding", MODEL_INPUT_LENGTH * D_MODEL),
    ("final LayerNorm", 2 * D_MODEL),
]

for name, count in parameter_rows:
    print(f"{name:30} {count:>10,}")
print("-" * 42)
print(f"{'total':30} {parameter_count:>10,}")

assert sum(count for _, count in parameter_rows) == 10_000_000
assert parameter_count == 10_000_000


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

block_labels = [
    "tokens + positions",
    "5 × decoder block",
    "final LayerNorm",
    "tied token head",
]
for index, label in enumerate(block_labels):
    x = index * 2.2
    axes[0].add_patch(plt.Rectangle((x, 0.25), 1.7, 0.5, fill=False))
    axes[0].text(x + 0.85, 0.5, label, ha="center", va="center", fontsize=9)
    if index < len(block_labels) - 1:
        axes[0].annotate(
            "",
            xy=(x + 2.15, 0.5),
            xytext=(x + 1.7, 0.5),
            arrowprops={"arrowstyle": "->"},
        )
axes[0].set_xlim(0, 8.5)
axes[0].set_ylim(0, 1)
axes[0].axis("off")
axes[0].set_title("Exact 10M decoder-only transformer")

axes[1].imshow(np.asarray(CAUSAL_MASK), cmap="Blues", interpolation="nearest")
axes[1].set_xlabel("key position")
axes[1].set_ylabel("query position")
axes[1].set_title("Causal attention mask")

plt.tight_layout()
plt.show()


## Train


In [ ]:
ANSWER_MASK = jnp.arange(MODEL_INPUT_LENGTH) >= MODEL_INPUT_LENGTH - ANSWER_DIGITS

def answer_loss(logits, targets):
    log_probabilities = jax.nn.log_softmax(logits.astype(jnp.float32), axis=-1)
    token_losses = -jnp.take_along_axis(
        log_probabilities,
        targets[..., None],
        axis=-1,
    ).squeeze(-1)

    denominator = targets.shape[0] * ANSWER_DIGITS
    loss = jnp.sum(jnp.where(ANSWER_MASK[None, :], token_losses, 0.0)) / denominator

    predictions = jnp.argmax(logits, axis=-1)
    correct = predictions == targets
    accuracy = jnp.sum(jnp.where(ANSWER_MASK[None, :], correct, False)) / denominator
    return loss, accuracy

graphdef, params = nnx.split(model, nnx.Param)

def path_parts(path):
    return tuple(
        str(getattr(entry, "key", getattr(entry, "idx", entry)))
        for entry in path
    )

def should_decay(path, leaf):
    parts = path_parts(path)
    return (
        leaf.ndim == 2
        and "kernel" in parts
        and ("attention" in parts or "ffn" in parts)
    )

decay_mask = jax.tree_util.tree_map_with_path(should_decay, params)

learning_rate = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=PEAK_LR,
    warmup_steps=WARMUP_STEPS,
    decay_steps=MAX_STEPS,
    end_value=FINAL_LR,
)

optimizer = optax.chain(
    optax.clip_by_global_norm(GRAD_CLIP),
    optax.scale_by_adam(b1=0.9, b2=0.99, eps=1e-8),
    optax.masked(optax.add_decayed_weights(WEIGHT_DECAY), decay_mask),
    optax.scale_by_learning_rate(learning_rate),
)
optimizer_state = optimizer.init(params)

def all_finite(tree):
    return jnp.all(
        jnp.stack([jnp.all(jnp.isfinite(leaf)) for leaf in jax.tree.leaves(tree)])
    )

@jax.jit
def train_step(params, optimizer_state, inputs, targets):
    def objective(candidate_params):
        current_model = nnx.merge(graphdef, candidate_params)
        return answer_loss(current_model(inputs), targets)

    (loss, accuracy), gradients = jax.value_and_grad(
        objective, has_aux=True
    )(params)

    gradient_norm = optax.global_norm(gradients)
    updates, new_optimizer_state = optimizer.update(
        gradients, optimizer_state, params
    )
    new_params = optax.apply_updates(params, updates)

    finite = (
        jnp.isfinite(loss)
        & all_finite(gradients)
        & all_finite(new_params)
    )

    safe_params = jax.tree.map(
        lambda new, old: jnp.where(finite, new, old),
        new_params,
        params,
    )
    safe_optimizer_state = jax.tree.map(
        lambda new, old: jnp.where(finite, new, old),
        new_optimizer_state,
        optimizer_state,
    )

    metrics = {
        "loss": loss,
        "answer_token_accuracy": accuracy,
        "gradient_norm": gradient_norm,
        "finite": finite,
    }
    return safe_params, safe_optimizer_state, metrics


In [ ]:
def greedy_generate(current_model, prompts):
    buffer = jnp.zeros(
        (prompts.shape[0], MODEL_INPUT_LENGTH),
        dtype=jnp.int32,
    )
    buffer = buffer.at[:, :PROMPT_LENGTH].set(prompts)

    def generate_digit(current_buffer, offset):
        logits = current_model(current_buffer)
        position = PROMPT_LENGTH + offset - 1
        next_token = jnp.argmax(logits[:, position, :], axis=-1).astype(jnp.int32)

        current_buffer = jax.lax.cond(
            offset < ANSWER_DIGITS - 1,
            lambda value: value.at[:, PROMPT_LENGTH + offset].set(next_token),
            lambda value: value,
            current_buffer,
        )
        return current_buffer, next_token

    _, generated = jax.lax.scan(
        generate_digit,
        buffer,
        jnp.arange(ANSWER_DIGITS),
    )
    generated = jnp.swapaxes(generated, 0, 1)
    valid = jnp.all(generated < 10, axis=-1)
    return generated, valid

@jax.jit
def evaluate_teacher_forced(params, inputs, targets):
    current_model = nnx.merge(graphdef, params)
    return answer_loss(current_model(inputs), targets)

@jax.jit
def generate_from_params(params, prompts):
    current_model = nnx.merge(graphdef, params)
    return greedy_generate(current_model, prompts)

def evaluate_validation(params):
    loss_total = 0.0
    token_accuracy_total = 0.0
    exact_total = 0

    for start in range(0, len(validation_ids), EVAL_BATCH_SIZE):
        ids = validation_ids[start : start + EVAL_BATCH_SIZE]
        sequences = make_sequences(ids).astype(np.int32)
        inputs = jnp.asarray(sequences[:, :-1])
        targets = jnp.asarray(sequences[:, 1:])

        loss, token_accuracy = evaluate_teacher_forced(params, inputs, targets)
        generated, valid = generate_from_params(
            params,
            inputs[:, :PROMPT_LENGTH],
        )

        expected = sequences[:, -ANSWER_DIGITS:]
        generated = np.asarray(generated)
        valid = np.asarray(valid)
        correct = valid & np.all(generated == expected, axis=1)

        loss_total += float(loss) * len(ids)
        token_accuracy_total += float(token_accuracy) * len(ids)
        exact_total += int(correct.sum())

    return {
        "loss": loss_total / len(validation_ids),
        "token_accuracy": token_accuracy_total / len(validation_ids),
        "exact_match": exact_total / len(validation_ids),
    }

steps_for_plot = np.arange(MAX_STEPS)
plt.figure(figsize=(7, 3))
plt.plot(steps_for_plot, [float(learning_rate(step)) for step in steps_for_plot])
plt.xlabel("step")
plt.ylabel("learning rate")
plt.title("AdamW learning-rate schedule")
plt.show()


## One-step compatibility smoke test

This uses the real dataset, exact 10,000,000-parameter model, optimizer, loss,
checkpoint format, and greedy generation path. It updates a temporary parameter
tree exactly once, then discards it; the complete training state below remains
unchanged.


In [ ]:
SMOKE_BATCH_SIZE = 8
_smoke_inputs, _smoke_targets = batcher.sample()
_smoke_inputs = jnp.asarray(_smoke_inputs[:SMOKE_BATCH_SIZE])
_smoke_targets = jnp.asarray(_smoke_targets[:SMOKE_BATCH_SIZE])

assert _smoke_inputs.shape == (SMOKE_BATCH_SIZE, MODEL_INPUT_LENGTH)
assert _smoke_targets.shape == (SMOKE_BATCH_SIZE, MODEL_INPUT_LENGTH)
assert parameter_count == 10_000_000

# Exactly one optimizer update on a temporary tree.
_smoke_params, _smoke_optimizer_state, _smoke_metrics = train_step(
    params,
    optimizer_state,
    _smoke_inputs,
    _smoke_targets,
)
jax.block_until_ready(_smoke_metrics["loss"])
assert bool(jax.device_get(_smoke_metrics["finite"]))

_smoke_eval_loss, _smoke_eval_accuracy = evaluate_teacher_forced(
    _smoke_params,
    _smoke_inputs,
    _smoke_targets,
)
_smoke_checkpoint_bytes = pickle.dumps(
    {
        "params": jax.device_get(_smoke_params),
        "step": 1,
        "validation": {
            "loss": float(_smoke_eval_loss),
            "token_accuracy": float(_smoke_eval_accuracy),
        },
        "parameter_count": parameter_count,
    }
)
_smoke_checkpoint = pickle.loads(_smoke_checkpoint_bytes)
assert set(_smoke_checkpoint) == {
    "params",
    "step",
    "validation",
    "parameter_count",
}
assert _smoke_checkpoint["parameter_count"] == 10_000_000

_smoke_restored_params = jax.tree.map(jnp.asarray, _smoke_checkpoint["params"])
_smoke_restored_model = nnx.merge(graphdef, _smoke_restored_params)
_smoke_logits = _smoke_restored_model(_smoke_inputs)
assert _smoke_logits.shape == (
    SMOKE_BATCH_SIZE,
    MODEL_INPUT_LENGTH,
    VOCAB_SIZE,
)
_smoke_generated, _smoke_valid = generate_from_params(
    _smoke_restored_params,
    _smoke_inputs[:, :PROMPT_LENGTH],
)
assert _smoke_generated.shape == (SMOKE_BATCH_SIZE, ANSWER_DIGITS)
assert _smoke_valid.shape == (SMOKE_BATCH_SIZE,)

print(
    "One-step smoke test passed | "
    f"loss {float(_smoke_metrics['loss']):.4f} | "
    f"evaluation loss {float(_smoke_eval_loss):.4f} | "
    f"generated shape {tuple(_smoke_generated.shape)}"
)

del (
    _smoke_params,
    _smoke_optimizer_state,
    _smoke_checkpoint_bytes,
    _smoke_checkpoint,
    _smoke_restored_params,
    _smoke_restored_model,
    _smoke_logits,
    _smoke_generated,
    _smoke_valid,
)


In [ ]:
for artifact in [CHECKPOINT_PATH, HISTORY_PATH, FAILURES_PATH, RESULTS_PATH]:
    artifact.unlink(missing_ok=True)

history = []
validation_history = []
best_exact_match = -1.0
best_validation_loss = math.inf
perfect_validation_streak = 0

compile_started = time.perf_counter()
inputs, targets = batcher.sample()
params, optimizer_state, metrics = train_step(
    params,
    optimizer_state,
    jnp.asarray(inputs),
    jnp.asarray(targets),
)
jax.block_until_ready(metrics["loss"])
compile_seconds = time.perf_counter() - compile_started
first_metrics = jax.device_get(metrics)
assert bool(first_metrics["finite"]), "Non-finite values in the first step."
print(
    f"Compilation + first step: {compile_seconds:.2f}s | "
    f"loss {float(first_metrics['loss']):.4f} | "
    f"token acc {100 * float(first_metrics['answer_token_accuracy']):.2f}%"
)

training_started = time.perf_counter()
last_log_time = training_started
last_log_step = 1
final_step = 1

for step in range(2, MAX_STEPS + 1):
    inputs, targets = batcher.sample()
    params, optimizer_state, metrics = train_step(
        params,
        optimizer_state,
        jnp.asarray(inputs),
        jnp.asarray(targets),
    )
    final_step = step

    if step % LOG_EVERY == 0:
        host_metrics = jax.device_get(metrics)
        now = time.perf_counter()
        elapsed = now - last_log_time
        completed = step - last_log_step
        examples_per_second = completed * BATCH_SIZE / elapsed

        record = {
            "step": step,
            "loss": float(host_metrics["loss"]),
            "answer_token_accuracy": float(host_metrics["answer_token_accuracy"]),
            "gradient_norm": float(host_metrics["gradient_norm"]),
            "learning_rate": float(learning_rate(step - 1)),
            "examples_per_second": examples_per_second,
        }
        assert bool(host_metrics["finite"]), f"Non-finite values at step {step}"
        history.append(record)

        print(
            f"step {step:4d} | "
            f"loss {record['loss']:.4f} | "
            f"token acc {100 * record['answer_token_accuracy']:6.2f}% | "
            f"grad {record['gradient_norm']:.3f} | "
            f"{examples_per_second:,.0f} examples/s"
        )

        last_log_time = now
        last_log_step = step

    if step % EVAL_EVERY == 0:
        validation = evaluate_validation(params)
        validation["step"] = step
        validation_history.append(validation)

        print(
            f"validation | loss {validation['loss']:.4f} | "
            f"token acc {100 * validation['token_accuracy']:6.2f}% | "
            f"exact match {100 * validation['exact_match']:6.2f}%"
        )

        improved = (
            validation["exact_match"] > best_exact_match
            or (
                validation["exact_match"] == best_exact_match
                and validation["loss"] < best_validation_loss
            )
        )

        if improved:
            best_exact_match = validation["exact_match"]
            best_validation_loss = validation["loss"]
            checkpoint = {
                "params": jax.device_get(params),
                "step": step,
                "validation": validation,
                "parameter_count": 10_000_000,
            }
            CHECKPOINT_PATH.write_bytes(pickle.dumps(checkpoint))
            print("saved best checkpoint")

        perfect_validation_streak = (
            perfect_validation_streak + 1
            if validation["exact_match"] == 1.0
            else 0
        )
        if perfect_validation_streak == 2:
            print("Validation reached 100% twice. Training stopped.")
            break

training_seconds = time.perf_counter() - training_started
HISTORY_PATH.write_text(
    json.dumps(
        {
            "compile_seconds": compile_seconds,
            "training_seconds": training_seconds,
            "final_step": final_step,
            "train": history,
            "validation": validation_history,
        },
        indent=2,
    )
)

print(f"Training time after compilation: {training_seconds / 60:.2f} minutes")
print(f"Best validation exact match: {100 * best_exact_match:.4f}%")


## Curves


In [ ]:
train_steps = [row["step"] for row in history]
validation_steps = [row["step"] for row in validation_history]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

axes[0, 0].plot(train_steps, [row["loss"] for row in history])
axes[0, 0].plot(
    validation_steps,
    [row["loss"] for row in validation_history],
    marker="o",
    label="validation",
)
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("Answer loss")
axes[0, 0].set_xlabel("step")
axes[0, 0].legend()

axes[0, 1].plot(
    train_steps,
    [100 * row["answer_token_accuracy"] for row in history],
)
axes[0, 1].set_title("Training answer-token accuracy")
axes[0, 1].set_xlabel("step")
axes[0, 1].set_ylabel("%")

axes[1, 0].plot(
    validation_steps,
    [100 * row["exact_match"] for row in validation_history],
    marker="o",
)
axes[1, 0].set_title("Validation greedy exact match")
axes[1, 0].set_xlabel("step")
axes[1, 0].set_ylabel("%")

axes[1, 1].plot(
    train_steps,
    [row["gradient_norm"] for row in history],
)
axes[1, 1].set_title("Gradient norm")
axes[1, 1].set_xlabel("step")

plt.tight_layout()
plt.show()


## Evaluate


In [ ]:
checkpoint = pickle.loads(CHECKPOINT_PATH.read_bytes())
params = jax.tree.map(jnp.asarray, checkpoint["params"])
trained_model = nnx.merge(graphdef, params)

print("Restored checkpoint from step", checkpoint["step"])
print("Validation exact match", checkpoint["validation"]["exact_match"])


In [ ]:
def evaluate_complete_domain(params):
    predictions = np.empty((1_000_000, ANSWER_DIGITS), dtype=np.uint8)
    valid = np.empty(1_000_000, dtype=bool)

    started = time.perf_counter()

    for start in range(0, 1_000_000, EVAL_BATCH_SIZE):
        ids = np.arange(start, start + EVAL_BATCH_SIZE, dtype=np.int32)
        sequences = make_sequences(ids).astype(np.int32)
        prompts = jnp.asarray(sequences[:, :PROMPT_LENGTH])
        generated, batch_valid = generate_from_params(params, prompts)
        predictions[start : start + EVAL_BATCH_SIZE] = np.asarray(generated)
        valid[start : start + EVAL_BATCH_SIZE] = np.asarray(batch_valid)

    pair_ids = np.arange(1_000_000, dtype=np.int32)
    a, b = pair_ids_to_operands(pair_ids)
    total = a + b
    expected = np.stack(
        [
            total % 10,
            (total // 10) % 10,
            (total // 100) % 10,
            (total // 1000) % 10,
        ],
        axis=1,
    ).astype(np.uint8)

    correct = valid & np.all(predictions == expected, axis=1)

    split_labels = np.empty(1_000_000, dtype=np.int8)
    split_labels[train_ids] = 0
    split_labels[validation_ids] = 1
    split_labels[test_ids] = 2

    split_names = ["train", "validation", "test"]
    split_results = {}
    for split_index, split_name in enumerate(split_names):
        mask = split_labels == split_index
        split_results[split_name] = {
            "correct": int(correct[mask].sum()),
            "total": int(mask.sum()),
            "exact_match": float(correct[mask].mean()),
        }

    carry_results = {}
    codes = carry_codes(a, b)
    for code_value in range(8):
        mask = codes == code_value
        carry_results[f"{code_value:03b}"] = float(correct[mask].mean())

    length_results = {}
    lengths_a = operand_lengths(a)
    lengths_b = operand_lengths(b)
    for length_a in range(1, 4):
        for length_b in range(1, 4):
            mask = (lengths_a == length_a) & (lengths_b == length_b)
            length_results[f"{length_a}×{length_b}"] = float(correct[mask].mean())

    swapped_ids = 1000 * b + a
    commutative = (
        valid
        & valid[swapped_ids]
        & np.all(predictions == predictions[swapped_ids], axis=1)
    )

    failure_indices = np.flatnonzero(~correct)
    with FAILURES_PATH.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(
            [
                "pair_id",
                "a",
                "b",
                "target",
                "prediction",
                "internal_digits",
                "valid_digits",
                "split",
                "carry_pattern",
            ]
        )
        split_text = np.asarray(split_names)
        for index in failure_indices:
            internal = "".join(str(int(value)) for value in predictions[index])
            prediction = (
                internal[::-1].lstrip("0") or "0"
                if valid[index]
                else "<invalid>"
            )
            writer.writerow(
                [
                    int(index),
                    int(a[index]),
                    int(b[index]),
                    int(total[index]),
                    prediction,
                    internal,
                    bool(valid[index]),
                    split_text[split_labels[index]],
                    f"{codes[index]:03b}",
                ]
            )

    results = {
        "parameter_count": 10_000_000,
        "checkpoint_step": int(checkpoint["step"]),
        "evaluation_seconds": time.perf_counter() - started,
        "overall": {
            "correct": int(correct.sum()),
            "total": 1_000_000,
            "exact_match": float(correct.mean()),
            "failures": int((~correct).sum()),
            "invalid_generations": int((~valid).sum()),
        },
        "splits": split_results,
        "carry_patterns": carry_results,
        "operand_lengths": length_results,
        "commutativity_consistency": float(commutative.mean()),
    }

    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    return results, correct, predictions, valid

results, correct, all_predictions, all_valid = evaluate_complete_domain(params)
print(json.dumps(results, indent=2))
print("Failures written to", FAILURES_PATH)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

carry_labels = list(results["carry_patterns"])
axes[0].bar(
    carry_labels,
    [100 * results["carry_patterns"][label] for label in carry_labels],
)
axes[0].set_ylim(0, 100.5)
axes[0].set_xlabel("carry pattern: units, tens, hundreds")
axes[0].set_ylabel("exact match %")
axes[0].set_title("Accuracy by carry pattern")

length_labels = list(results["operand_lengths"])
axes[1].bar(
    length_labels,
    [100 * results["operand_lengths"][label] for label in length_labels],
)
axes[1].set_ylim(0, 100.5)
axes[1].set_xlabel("operand lengths")
axes[1].set_ylabel("exact match %")
axes[1].set_title("Accuracy by operand length")

plt.tight_layout()
plt.show()


## Use the model


In [ ]:
def predict_pair(a, b):
    assert 0 <= a <= 999 and 0 <= b <= 999
    prompt = jnp.asarray(encode(format_prompt(a, b))[None, :])
    generated, valid = generate_from_params(params, prompt)
    generated = np.asarray(generated[0])
    assert bool(np.asarray(valid[0])), "The model generated a non-digit token."
    return decode_internal_answer(generated)

for a, b in [
    (0, 0),
    (1, 9),
    (99, 1),
    (123, 456),
    (347, 928),
    (500, 500),
    (999, 1),
    (999, 999),
]:
    print(f"{a} + {b} = {predict_pair(a, b)}")


In [ ]:
expression = input("addition> ").strip()
match = re.fullmatch(r"(\d{1,3})\s*\+\s*(\d{1,3})\s*=?", expression)
assert match is not None, "Use a form such as 123 + 456"
a, b = int(match.group(1)), int(match.group(2))
print(predict_pair(a, b))


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/jax_addition_run",
    "zip",
    root_dir=RUN_DIR,
)

print("Checkpoint:", CHECKPOINT_PATH)
print("History:", HISTORY_PATH)
print("Results:", RESULTS_PATH)
print("Failures:", FAILURES_PATH)
print("Archive:", archive)

files.download(archive)
